load dataset

In [3]:
import json
import pandas as pd

with open("/home/jovyan/Intern Assessment/ZW/RecruitView.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Records loaded:", len(df))

Records loaded: 2011


Q1) How many interview video clips are in the dataset?

In [4]:
video_count = len(df)

print("Q1 Answer")
print("Total interview video clips:", video_count)

Q1 Answer
Total interview video clips: 2011


Q2) How many participants contributed to the dataset?

In [5]:
participant_count = df["user_no"].nunique()

print("Q2 Answer")
print("Total participants:", participant_count)

Q2 Answer
Total participants: 331


Q3) How many interview questions are available?

In [6]:
question_count = df["question_id"].nunique()

print("Q3 Answer")
print("Total interview questions:", question_count)

Q3 Answer
Total interview questions: 76


In [ ]:
brief view of sample questions

In [7]:
questions = (
    df[["question_id", "question"]]
    .drop_duplicates()
    .sort_values("question_id")
)

questions

,question_id,question
0,1,Introduce yourself
385,10,Can you give an example of a time when you suc...
403,11,Tell me about a time when you had to step into...
420,12,How do you respond to change?
445,13,What was the toughest decision you ever had to...
...,...,...
1940,74,Can you give an example of a time when you coa...
1958,75,Share a story of a time when you rallied your ...
193,76,Tell me about yourself
1976,8,"If you won a Rs.10-crore lottery, would you st..."


Q4) What are the input modalities used by the system?

In [8]:
print(df.columns.tolist())

['id', 'video', 'duration', 'question_id', 'question', 'video_quality', 'user_no', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'overall_personality', 'interview_score', 'answer_score', 'speaking_skills', 'confidence_score', 'facial_expression', 'overall_performance', 'transcript', 'gemini_summary']


In [10]:
modalities = {
    "Visual": [],
    "Audio/Speech": [],
    "Text": []
}

for col in df.columns:
    col_lower = col.lower()

    # Visual-related features
    if any(x in col_lower for x in [
        "video", "facial"
    ]):
        modalities["Visual"].append(col)

    # Audio/Speech-related features
    elif any(x in col_lower for x in [
        "speaking", "confidence", "duration"
    ]):
        modalities["Audio/Speech"].append(col)

    # Text-related features
    elif any(x in col_lower for x in [
        "question", "transcript", "summary"
    ]):
        modalities["Text"].append(col)

for modality, cols in modalities.items():
    print(f"\n{modality} Modality")
    print("-" * 30)
    for c in cols:
        print(c)


Visual Modality
------------------------------
video
video_quality
facial_expression

Audio/Speech Modality
------------------------------
duration
speaking_skills
confidence_score

Text Modality
------------------------------
question_id
question
transcript
gemini_summary


Q5) What are the output scores predicted by the model?

In [11]:
df.columns.tolist()

['id',
 'video',
 'duration',
 'question_id',
 'question',
 'video_quality',
 'user_no',
 'openness',
 'conscientiousness',
 'extraversion',
 'agreeableness',
 'neuroticism',
 'overall_personality',
 'interview_score',
 'answer_score',
 'speaking_skills',
 'confidence_score',
 'facial_expression',
 'overall_performance',
 'transcript',
 'gemini_summary']

In [16]:
score_columns = []

for col in df.columns:
    if df[col].dtype in ["float64", "int64"]:
        score_columns.append(col)

print("Numeric Score Columns:")
for col in score_columns:
    print("-", col)

print("\nTotal Score Columns:", len(score_columns))

Numeric Score Columns:
- openness
- conscientiousness
- extraversion
- agreeableness
- neuroticism
- overall_personality
- interview_score
- answer_score
- speaking_skills
- confidence_score
- facial_expression
- overall_performance

Total Score Columns: 12


Q6) Video Duration Distribution

In [17]:
duration_counts = df["duration"].value_counts()

duration_summary = duration_counts.reset_index()
duration_summary.columns = ["Duration Category", "Count"]

duration_summary["Percentage (%)"] = (
    duration_summary["Count"] / len(df) * 100
).round(2)

print("Video Duration Distribution")
print(duration_summary)

print("\nTotal Videos:", len(df))

Video Duration Distribution
  Duration Category  Count  Percentage (%)
0            medium    754           37.49
1             short    680           33.81
2              long    577           28.69

Total Videos: 2011


Q7) Video Quality Distribution

In [18]:
quality_counts = df["video_quality"].value_counts()

quality_summary = quality_counts.reset_index()
quality_summary.columns = ["Video Quality", "Count"]

quality_summary["Percentage (%)"] = (
    quality_summary["Count"] / len(df) * 100
).round(2)

print("Video Quality Distribution")
print(quality_summary)

print("\nTotal Videos:", len(df))

Video Quality Distribution
  Video Quality  Count  Percentage (%)
0          High   1675           83.29
1           Low    336           16.71

Total Videos: 2011


Q8) What is the distribution of Overall Performance scores?

In [19]:
print(df["overall_performance"].describe())

count    2.011000e+03
mean     2.270578e-10
std      1.200662e+00
min     -9.311059e+00
25%     -5.748997e-01
50%      4.342030e-02
75%      6.014077e-01
max      8.848990e+00
Name: overall_performance, dtype: float64


Q9) How are candidates distributed by performance level?

In [20]:
def performance_category(score):
    if score < -0.33:
        return "Low"
    elif score < 0.33:
        return "Medium"
    else:
        return "High"

df["performance_level"] = df["overall_performance"].apply(performance_category)

performance_counts = df["performance_level"].value_counts()

performance_summary = performance_counts.reset_index()
performance_summary.columns = ["Performance Level", "Count"]

performance_summary["Percentage (%)"] = (
    performance_summary["Count"] / len(df) * 100
).round(2)

print(performance_summary)

  Performance Level  Count  Percentage (%)
0              High    731           36.35
1               Low    684           34.01
2            Medium    596           29.64


Q10. Which interview question has the highest average overall performance?

In [24]:
# Calculate average overall performance for each question

question_performance = (
    df.groupby(["question_id", "question"])["overall_performance"]
      .mean()
      .reset_index()
      .sort_values("overall_performance", ascending=False)
)

# Top 3 Questions
top3 = question_performance.head(3)

print("=" * 80)
print("TOP 3 INTERVIEW QUESTIONS WITH HIGHEST OVERALL PERFORMANCE")
print("=" * 80)

for rank, (_, row) in enumerate(top3.iterrows(), start=1):
    print(f"\nRank {rank}")
    print(f"Question ID      : {row['question_id']}")
    print(f"Question         : {row['question']}")
    print(f"Average Score    : {row['overall_performance']:.12f}")

print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)

print(
    top3[["question_id", "question", "overall_performance"]]
    .rename(columns={
        "question_id": "Question ID",
        "question": "Question",
        "overall_performance": "Average Performance Score"
    })
    .to_string(index=False)
)

TOP 3 INTERVIEW QUESTIONS WITH HIGHEST OVERALL PERFORMANCE

Rank 1
Question ID      : 74
Question         : Can you give an example of a time when you coached or mentored a colleague to help them achieve their goals?
Average Score    : 0.000000013454

Rank 2
Question ID      : 26
Question         : Can you give an example of a time when you had to take the initiative to solve a problem without being asked?
Average Score    : 0.000000008845

Rank 3
Question ID      : 11
Question         : Tell me about a time when you had to step into a role outside of your expertise to support the team's objectives.
Average Score    : 0.000000008317

SUMMARY TABLE
Question ID                                                                                                          Question  Average Performance Score
         74      Can you give an example of a time when you coached or mentored a colleague to help them achieve their goals?               1.345429e-08
         26     Can you give an exampl

Q11) Does video quality affect overall performance?

In [22]:
quality_performance = (
    df.groupby("video_quality")["overall_performance"]
    .mean()
    .sort_values(ascending=False)
)

print(quality_performance)

video_quality
Low     0.075916
High   -0.015228
Name: overall_performance, dtype: float64


In [27]:
# Top 10 Candidates by Overall Performance

top10_candidates = (
    df.sort_values("overall_performance", ascending=False)
      [["user_no", "id", "overall_performance", "question_id", "question"]]
      .head(10)
      .reset_index(drop=True)
)

print("=" * 120)
print("TOP 10 CANDIDATES BASED ON OVERALL PERFORMANCE")
print("=" * 120)

for rank, row in top10_candidates.iterrows():
    print(f"\nRank #{rank + 1}")
    print(f"User No           : {row['user_no']}")
    print(f"Video ID          : {row['id']}")
    print(f"Overall Score     : {row['overall_performance']:.10f}")
    print(f"Question ID       : {row['question_id']}")
    print(f"Question          : {row['question']}")
    print("-" * 120)

TOP 10 CANDIDATES BASED ON OVERALL PERFORMANCE

Rank #1
User No           : 301
Video ID          : 1773
Overall Score     : 8.8489897028
Question ID       : 65
Question          : Can you give an example of a situation where you successfully motivated a disengaged team member to contribute effectively to a project?
------------------------------------------------------------------------------------------------------------------------

Rank #2
User No           : 292
Video ID          : 1967
Overall Score     : 7.1626822361
Question ID       : 75
Question          : Share a story of a time when you rallied your team during a crisis, fostering resilience and determination.
------------------------------------------------------------------------------------------------------------------------

Rank #3
User No           : 215
Video ID          : 1064
Overall Score     : 6.8041685060
Question ID       : 34
Question          : Can you give an example of a time when you led by example to pro